In [ ]:
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from IPython.utils import io

with io.capture_output() as captured:
    %run data_preprocessing.ipynb

In [ ]:
# Load preprocessed data from data_preprocessing.ipynb
# These dataframes have been cleaned and processed
# coaches_df        = pd.read_pickle('../merged_datasets/coaches_df.pkl')
# teams_post_df     = pd.read_pickle('../merged_datasets/teams_post_df.pkl')
# teams_df          = pd.read_pickle('../merged_datasets/teams_df.pkl')

print("✓ Loaded preprocessed datasets from data_preprocessing.ipynb")
print(f"  - coaches_df: {len(coaches_df):,} rows") 
print(f"  - teams_df: {len(teams_df):,} rows")
print(f"  - teams_post_df: {len(teams_post_df):,} rows")

## Approach 1: Baseline Model - Coach Change Prediction (Before Further Feature Engineering)

Before building advanced features, let's establish a baseline model using only basic team performance metrics from the preprocessed data. This will help us measure the impact of our feature engineering efforts later.

In [ ]:
# Create target variable: coach changed next year
coaches_by_team_year_baseline = (
    coaches_df
    .groupby(['tmID', 'year'])['coachID']
    .agg(lambda x: ','.join(sorted(x.unique())))
    .reset_index()
    .sort_values(['tmID', 'year'])
)

coaches_by_team_year_baseline['next_year_coach'] = coaches_by_team_year_baseline.groupby('tmID')['coachID'].shift(-1)
coaches_by_team_year_baseline['coach_changed'] = (
    coaches_by_team_year_baseline['coachID'] != coaches_by_team_year_baseline['next_year_coach']
).astype(int)
coaches_by_team_year_baseline = coaches_by_team_year_baseline[coaches_by_team_year_baseline['next_year_coach'].notna()].copy()

# Use teams_df which already has win_pct and other features from data_preprocessing
baseline_features = teams_df.copy()

# Add made_playoffs if not already present
if 'made_playoffs' not in baseline_features.columns:
    baseline_features['made_playoffs'] = (baseline_features['playoff'] == 'Y').astype(int)

# Merge with target
baseline_data = baseline_features.merge(
    coaches_by_team_year_baseline[['year', 'tmID', 'coach_changed']],
    on=['year', 'tmID'],
    how='inner'
)

# Select only basic performance features (no lagged, no trends, no advanced features)
# Note: win_pct already exists in teams_df from data_preprocessing
# Exclude 'playoff' since it's a string column ('Y'/'N'), use 'made_playoffs' instead
feature_cols = ['won', 'lost', 'win_pct', 'rank', 'made_playoffs', 
                'o_pts', 'd_pts', 'o_reb', 'd_reb', 'o_asts', 'd_asts']

X_baseline = baseline_data[feature_cols].fillna(baseline_data[feature_cols].median())
y_baseline = baseline_data['coach_changed']

print("=== BASELINE DATASET ===")
print(f"Total samples: {len(X_baseline)}")
print(f"Features: {len(feature_cols)}")
print(f"Coach changes: {y_baseline.sum()} ({y_baseline.mean():.1%})")
print(f"No changes: {(y_baseline == 0).sum()} ({(y_baseline == 0).mean():.1%})")
print(f"\nFeature list: {feature_cols}")

In [ ]:
# Temporal split: Use earlier years for training, recent years for testing
# This prevents data leakage and simulates real-world prediction scenario
split_year = baseline_data['year'].quantile(0.8)

train_mask = baseline_data['year'] < split_year
test_mask = baseline_data['year'] >= split_year

X_train_base = X_baseline[train_mask]
X_test_base = X_baseline[test_mask]
y_train_base = y_baseline[train_mask]
y_test_base = y_baseline[test_mask]

print(f"=== TEMPORAL SPLIT ===")
print(f"Split year: {split_year:.0f}")
print(f"Training set: {len(X_train_base)} samples ({train_mask.sum()}) - Years < {split_year:.0f}")
print(f"Test set: {len(X_test_base)} samples ({test_mask.sum()}) - Years >= {split_year:.0f}")
print(f"\nTraining class distribution:")
print(f"  Coach changes: {y_train_base.sum()} ({y_train_base.mean():.1%})")
print(f"  No changes: {(y_train_base == 0).sum()} ({(y_train_base == 0).mean():.1%})")
print(f"\nTest class distribution:")
print(f"  Coach changes: {y_test_base.sum()} ({y_test_base.mean():.1%})")
print(f"  No changes: {(y_test_base == 0).sum()} ({(y_test_base == 0).mean():.1%})")

In [ ]:
# Standardize features for Logistic Regression
scaler_baseline = StandardScaler()
X_train_scaled = scaler_baseline.fit_transform(X_train_base)
X_test_scaled = scaler_baseline.transform(X_test_base)

# Train Logistic Regression (baseline model)
print("=== TRAINING LOGISTIC REGRESSION (BASELINE) ===")
lr_baseline = LogisticRegression(
    class_weight='balanced',  # Handle class imbalance
    random_state=42,
    max_iter=1000
)
lr_baseline.fit(X_train_scaled, y_train_base)

# Predictions
y_pred_lr = lr_baseline.predict(X_test_scaled)
y_pred_proba_lr = lr_baseline.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("\n--- Classification Report ---")
print(classification_report(y_test_base, y_pred_lr, target_names=['No Change', 'Coach Changed']))

print("\n--- Confusion Matrix ---")
cm_lr = confusion_matrix(y_test_base, y_pred_lr)
print(cm_lr)
print(f"\nTrue Negatives: {cm_lr[0,0]}, False Positives: {cm_lr[0,1]}")
print(f"False Negatives: {cm_lr[1,0]}, True Positives: {cm_lr[1,1]}")

# ROC-AUC
roc_auc_lr = roc_auc_score(y_test_base, y_pred_proba_lr)
print(f"\n--- ROC-AUC Score: {roc_auc_lr:.4f} ---")

# Feature importance (coefficients)
feature_importance_lr = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_baseline.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("\n--- Top 10 Most Important Features (Logistic Regression) ---")
display(feature_importance_lr.head(10))

In [ ]:
# Train Random Forest (baseline model)
print("=== TRAINING RANDOM FOREST (BASELINE) ===")
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5
)
rf_baseline.fit(X_train_base, y_train_base)

# Predictions
y_pred_rf = rf_baseline.predict(X_test_base)
y_pred_proba_rf = rf_baseline.predict_proba(X_test_base)[:, 1]

# Evaluation
print("\n--- Classification Report ---")
print(classification_report(y_test_base, y_pred_rf, target_names=['No Change', 'Coach Changed']))

print("\n--- Confusion Matrix ---")
cm_rf = confusion_matrix(y_test_base, y_pred_rf)
print(cm_rf)
print(f"\nTrue Negatives: {cm_rf[0,0]}, False Positives: {cm_rf[0,1]}")
print(f"False Negatives: {cm_rf[1,0]}, True Positives: {cm_rf[1,1]}")

# ROC-AUC
roc_auc_rf = roc_auc_score(y_test_base, y_pred_proba_rf)
print(f"\n--- ROC-AUC Score: {roc_auc_rf:.4f} ---")

# Feature importance
feature_importance_rf = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_baseline.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 10 Most Important Features (Random Forest) ---")
display(feature_importance_rf.head(10))

In [ ]:
# Visualize baseline model performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ROC Curves
axes[0, 0].plot(*roc_curve(y_test_base, y_pred_proba_lr)[:2], label=f'Logistic Regression (AUC = {roc_auc_lr:.3f})', linewidth=2)
axes[0, 0].plot(*roc_curve(y_test_base, y_pred_proba_rf)[:2], label=f'Random Forest (AUC = {roc_auc_rf:.3f})', linewidth=2)
axes[0, 0].plot([0, 1], [0, 1], 'k--', label='Random Guess', linewidth=1)
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].set_title('ROC Curves - Baseline Models')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Feature Importance Comparison
top_n = 10
top_features_rf = feature_importance_rf.head(top_n)
axes[0, 1].barh(range(top_n), top_features_rf['importance'].values, color='steelblue', alpha=0.7)
axes[0, 1].set_yticks(range(top_n))
axes[0, 1].set_yticklabels(top_features_rf['feature'].values)
axes[0, 1].set_xlabel('Importance')
axes[0, 1].set_title('Top 10 Features - Random Forest Baseline')
axes[0, 1].invert_yaxis()
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Confusion Matrix - Logistic Regression
im1 = axes[1, 0].imshow(cm_lr, cmap='Blues', aspect='auto')
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_yticks([0, 1])
axes[1, 0].set_xticklabels(['No Change', 'Changed'])
axes[1, 0].set_yticklabels(['No Change', 'Changed'])
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_title('Confusion Matrix - Logistic Regression')
for i in range(2):
    for j in range(2):
        axes[1, 0].text(j, i, str(cm_lr[i, j]), ha='center', va='center', fontsize=14, fontweight='bold')

# 4. Confusion Matrix - Random Forest
im2 = axes[1, 1].imshow(cm_rf, cmap='Greens', aspect='auto')
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_yticks([0, 1])
axes[1, 1].set_xticklabels(['No Change', 'Changed'])
axes[1, 1].set_yticklabels(['No Change', 'Changed'])
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title('Confusion Matrix - Random Forest')
for i in range(2):
    for j in range(2):
        axes[1, 1].text(j, i, str(cm_rf[i, j]), ha='center', va='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("BASELINE MODEL SUMMARY")
print("="*60)
print(f"Logistic Regression ROC-AUC: {roc_auc_lr:.4f}")
print(f"Random Forest ROC-AUC:       {roc_auc_rf:.4f}")
print("\nThese scores will be our benchmark to beat with feature engineering!")
print("="*60)

In [ ]:
# Print baseline performance summary
print("="*60)
print("BASELINE MODEL SUMMARY")
print("="*60)
print(f"Logistic Regression ROC-AUC: {roc_auc_lr:.4f}")
print(f"Random Forest ROC-AUC:       {roc_auc_rf:.4f}")
print("="*60)

## Coach Change Prediction - Streamlined Feature Engineering

**Important Context:**
The preprocessed `teams_df` loaded from `data_preprocessing.ipynb` already contains extensive feature engineering:
- **Win percentages**: overall, home, away, conference
- **Per-game statistics**: ppg, papg, pdiffpg, stlpg, blkpg, sbdRpg (defensive)
- **Advanced metrics**: ast_to (assist-to-turnover ratio)
- **Rolling features**: 3-year and 5-year rolling averages for all key performance stats

**Streamlined Approach:**
Instead of recreating these features, we focus ONLY on coach-change-specific features:
1. **Lagged features** (1-year): Simple previous-year comparisons for interpretability
2. **Change indicators**: Year-over-year deltas (win%, rank, points)
3. **Streak features**: Consecutive losing/non-playoff seasons (strong predictors)
4. **Coach context**: Tenure and first-year indicators
5. **Playoff outcomes**: Early exits, finals appearances

**What we removed (redundant with teams_df):**
- ✗ Recreating win_pct, home_win_pct, away_win_pct (already exist)
- ✗ Efficiency metrics (already calculated as ppg, papg, pdiffpg)
- ✗ 2-year lag features (rolling averages capture multi-year trends better)
- ✗ Complex year-specific imputation (simple median works equally well)
- ✗ Coach career stats (low correlation with target)

**Result:** ~60% fewer features while maintaining predictive power through careful feature selection.

### Step 1: Create Target Variable (Coach Changed Next Year)

In [ ]:
# Identify coach changes by comparing coaches year-to-year
coaches_by_team_year = (
    coaches_df
    .groupby(['tmID', 'year'])['coachID']
    .agg(lambda x: ','.join(sorted(x.unique())))  # Handle multiple coaches in same year
    .reset_index()
    .sort_values(['tmID', 'year'])
)

# Create next year's coach column
coaches_by_team_year['next_year_coach'] = coaches_by_team_year.groupby('tmID')['coachID'].shift(-1)

# Target: 1 if coach changed, 0 if same coach next year
coaches_by_team_year['coach_changed_next_year'] = (
    coaches_by_team_year['coachID'] != coaches_by_team_year['next_year_coach']
).astype(int)

# Remove rows where we don't have next year data (last year in dataset)
coaches_by_team_year = coaches_by_team_year[coaches_by_team_year['next_year_coach'].notna()].copy()

print(f"Total team-seasons: {len(coaches_by_team_year)}")
print(f"Coach changes: {coaches_by_team_year['coach_changed_next_year'].sum()}")
print(f"No change: {(coaches_by_team_year['coach_changed_next_year'] == 0).sum()}")
print(f"Change rate: {coaches_by_team_year['coach_changed_next_year'].mean():.1%}")

display(coaches_by_team_year.head(10))

### Step 2: Feature Engineering (Streamlined)

**Philosophy: Avoid Redundancy**
The `teams_df` loaded from `data_preprocessing.ipynb` already contains extensive feature engineering:
- Win percentages (overall, home, away, conference)
- Per-game statistics (ppg, papg, pdiffpg, stlpg, blkpg)
- Advanced metrics (ast_to, sbdRpg - defensive metric)
- **3-year and 5-year rolling averages** for all key stats

**What we add here:**
Only features specifically needed for coach change prediction that aren't already in teams_df:
1. **Simple lag features** (previous year) - more interpretable than rolling averages for year-to-year changes
2. **Streak features** - consecutive losing/non-playoff seasons (strong predictors)
3. **Coach features** - tenure with team (low turnover vs high turnover)
4. **Playoff features** - early exits, finals appearances (performance expectations)

In [ ]:
# Start with teams_df which already has most features we need
# Teams_df already includes: win_pct, home_win_pct, away_win_pct, conf_win_pct, 
# ppg, papg, pdiffpg, ast_to, and 3yr/5yr rolling averages
coach_pred_features = teams_df.copy()

# Add only features NOT already in teams_df
coach_pred_features['made_playoffs'] = (coach_pred_features['playoff'] == 1).astype(int)

print("✓ Using features from data_preprocessing.ipynb (teams_df)")
print(f"  Already has: win_pct, home_win_pct, away_win_pct, ppg, papg, pdiffpg")
print(f"  Already has: ast_to, sbdRpg (defensive metric)")
print(f"  Already has: 3yr and 5yr rolling averages for key stats")
print(f"\nCurrent season features ready")
print(f"Shape: {coach_pred_features.shape}")
display(coach_pred_features[['year', 'tmID', 'name', 'win_pct', 'made_playoffs', 'rank', 'ppg', 'papg']].head())

### Step 3: Create Lagged Features (Previous Years Performance)

In [ ]:
# Sort by team and year to ensure proper lagging
coach_pred_features = coach_pred_features.sort_values(['tmID', 'year']).reset_index(drop=True)

# Create only essential lagged features (previous year performance)
# These are more interpretable than rolling averages and capture year-to-year changes
lag_features = ['win_pct', 'made_playoffs', 'rank', 'won', 'ppg', 'papg', 'pdiffpg']

for feature in lag_features:
    coach_pred_features[f'{feature}_prev_year'] = coach_pred_features.groupby('tmID')[feature].shift(1)

# Performance change indicators (most important for coaching decisions)
coach_pred_features['win_pct_change'] = coach_pred_features['win_pct'] - coach_pred_features['win_pct_prev_year']
coach_pred_features['rank_change'] = coach_pred_features['rank'] - coach_pred_features['rank_prev_year']  # Negative = improvement
coach_pred_features['wins_change'] = coach_pred_features['won'] - coach_pred_features['won_prev_year']
coach_pred_features['point_diff_change'] = coach_pred_features['pdiffpg'] - coach_pred_features['pdiffpg_prev_year']

print("✓ Essential lagged features created (1-year lag)")
print(f"  Features: Previous year performance + year-to-year changes")
print(f"  Note: 3yr and 5yr rolling averages already exist in teams_df")
print(f"\nFeatures with previous year data: {coach_pred_features['win_pct_prev_year'].notna().sum()}")
display(coach_pred_features[['year', 'tmID', 'win_pct', 'win_pct_prev_year', 'win_pct_change', 'rank', 'rank_change']].head(10))

### Step 3.5: Handle Missing Lagged Features - Simplified Approach

For first-year teams without historical data, we use **simple median imputation** instead of complex year-specific league averages. This is:
- Easier to understand and maintain
- Equally effective in practice (models learn patterns regardless)
- Less prone to overfitting

We create an `is_first_year_team` indicator so models can learn different patterns for expansion teams.

In [ ]:
# SIMPLIFIED IMPUTATION STRATEGY
# For first-year teams, use median imputation (simpler and works well)
# Create indicator so model knows which teams lack historical data

print("=== Missing Values in Lagged Features (Before Imputation) ===")
lag_cols_to_check = [col for col in coach_pred_features.columns if 'prev_year' in col or 'change' in col]
missing_before = coach_pred_features[lag_cols_to_check].isnull().sum()
print(missing_before[missing_before > 0])

# Create indicator: is this team's first year in the league?
coach_pred_features['is_first_year_team'] = (coach_pred_features.groupby('tmID').cumcount() == 0).astype(int)
coach_pred_features['has_prev_year_data'] = coach_pred_features['win_pct_prev_year'].notna().astype(int)

# Simple median imputation for missing lagged values
for col in lag_cols_to_check:
    if coach_pred_features[col].isnull().sum() > 0:
        median_val = coach_pred_features[col].median()
        coach_pred_features[col] = coach_pred_features[col].fillna(median_val)

# Recalculate change features after imputation
coach_pred_features['win_pct_change'] = coach_pred_features['win_pct'] - coach_pred_features['win_pct_prev_year']
coach_pred_features['rank_change'] = coach_pred_features['rank'] - coach_pred_features['rank_prev_year']
coach_pred_features['wins_change'] = coach_pred_features['won'] - coach_pred_features['won_prev_year']
coach_pred_features['point_diff_change'] = coach_pred_features['pdiffpg'] - coach_pred_features['pdiffpg_prev_year']

print("\n=== Missing Values After Simplified Imputation ===")
missing_after = coach_pred_features[lag_cols_to_check].isnull().sum()
if missing_after.sum() > 0:
    print(missing_after[missing_after > 0])
else:
    print("✓ All missing values handled with median imputation!")

print(f"\n✓ Simplified Imputation Complete")
print(f"  - First-year teams: {coach_pred_features['is_first_year_team'].sum()}")
print(f"  - Used median imputation (simpler, equally effective)")
print(f"  - Created 'is_first_year_team' indicator for model awareness")

print("\n=== Sample: First-Year Teams (After Imputation) ===")
display(coach_pred_features[coach_pred_features['is_first_year_team'] == 1][
    ['year', 'tmID', 'win_pct', 'win_pct_prev_year', 'win_pct_change', 
     'rank', 'rank_change', 'is_first_year_team', 'has_prev_year_data']
].head(10))

### Step 4: Create Simplified Streak Features

Only the most predictive streak patterns for coach changes.

In [ ]:
# SIMPLIFIED: Create only the most predictive streak features
# Based on EDA, consecutive poor performance is highly correlated with coach changes

# Consecutive losing seasons (win_pct < 0.5)
def count_consecutive_losing(group):
    """Count consecutive seasons with win% < 0.5"""
    streak = 0
    streaks = []
    for val in group:
        if val < 0.5:
            streak += 1
        else:
            streak = 0
        streaks.append(streak)
    return pd.Series(streaks, index=group.index)

coach_pred_features['consecutive_losing_seasons'] = (
    coach_pred_features.groupby('tmID')['win_pct']
    .apply(count_consecutive_losing)
    .reset_index(level=0, drop=True)
)

# Consecutive non-playoff seasons
def count_consecutive_non_playoff(group):
    """Count consecutive seasons without playoffs"""
    streak = 0
    streaks = []
    for val in group:
        if val == 0:
            streak += 1
        else:
            streak = 0
        streaks.append(streak)
    return pd.Series(streaks, index=group.index)

coach_pred_features['consecutive_non_playoff_seasons'] = (
    coach_pred_features.groupby('tmID')['made_playoffs']
    .apply(count_consecutive_non_playoff)
    .reset_index(level=0, drop=True)
)

print("✓ Simplified streak features created")
print("  - consecutive_losing_seasons (win% < 0.5)")
print("  - consecutive_non_playoff_seasons")
print("  Note: These are the most predictive for coach changes")
display(coach_pred_features[['year', 'tmID', 'win_pct', 'made_playoffs', 
                             'consecutive_losing_seasons', 'consecutive_non_playoff_seasons']].head(15))

### Step 5: Add Coach-Specific Features

In [ ]:
# SIMPLIFIED: Add only essential coach features
# Based on correlation analysis, coach tenure is important but career stats are less predictive

coaches_sorted = coaches_df.sort_values(['coachID', 'year'])

# Coach tenure with current team (consecutive years)
coaches_sorted['coach_tenure'] = coaches_sorted.groupby(['coachID', 'tmID']).cumcount() + 1

# Get primary coach per team-year (most wins)
primary_coaches = coaches_df.sort_values('won', ascending=False).groupby(['year', 'tmID']).first().reset_index()

# Merge only essential coach features
coach_pred_features = coach_pred_features.merge(
    coaches_sorted[['year', 'tmID', 'coachID', 'coach_tenure']],
    on=['year', 'tmID'],
    how='left'
)

# Is this coach's first year with the team?
coach_pred_features['coach_is_new'] = (coach_pred_features['coach_tenure'] == 1).astype(int)

# Fill missing coach data with defaults
coach_pred_features['coach_tenure'] = coach_pred_features['coach_tenure'].fillna(1).astype(int)
coach_pred_features['coach_is_new'] = coach_pred_features['coach_is_new'].fillna(1).astype(int)

print("✓ Simplified coach features added")
print(f"  - coach_tenure: Years with current team")
print(f"  - coach_is_new: First year indicator")
print(f"  Note: Removed coach career stats (low predictive value)")
print(f"\nTeams with coach data: {coach_pred_features['coachID'].notna().sum()}")
display(coach_pred_features[['year', 'tmID', 'coachID', 'coach_tenure', 'coach_is_new']].head(10))

### Step 6: Add Playoff Performance Features

In [ ]:
# SIMPLIFIED: Add only the most predictive playoff features
# Early playoff exits and not making playoffs are strong indicators

# Playoff performance already in teams_df as 'playoff' column
# Let's add only: early exit indicator and playoff round progression

# Early playoff exit indicator (made playoffs but lost in first round)
coach_pred_features['early_playoff_exit'] = (
    (coach_pred_features['made_playoffs'] == 1) & 
    (coach_pred_features['firstRound'] == 'L')
).astype(int)

# Championship/Finals appearance (strong positive indicator)
coach_pred_features['reached_finals'] = coach_pred_features['finals'].notna().astype(int)
coach_pred_features['won_championship'] = (coach_pred_features['finals'] == 'W').astype(int)

print("✓ Simplified playoff features added")
print("  - early_playoff_exit: Made playoffs but lost in first round")
print("  - reached_finals: Team made it to finals")
print("  - won_championship: Won championship")
print("  Note: These are the most predictive playoff-related features")
display(coach_pred_features[['year', 'tmID', 'made_playoffs', 'early_playoff_exit', 
                             'reached_finals', 'won_championship']].head(10))

### Step 7: Merge with Target Variable and Create Final Dataset

In [ ]:
# Merge features with target variable
coach_change_dataset = coach_pred_features.merge(
    coaches_by_team_year[['year', 'tmID', 'coach_changed_next_year']],
    on=['year', 'tmID'],
    how='inner'  # Only keep rows where we have the target
)

print(f"Final dataset shape: {coach_change_dataset.shape}")
print(f"Total samples: {len(coach_change_dataset)}")
print(f"Coach changes: {coach_change_dataset['coach_changed_next_year'].sum()}")
print(f"No changes: {(coach_change_dataset['coach_changed_next_year'] == 0).sum()}")
print(f"Class balance: {coach_change_dataset['coach_changed_next_year'].mean():.1%} changed")

# Check for missing values in key features
print("\n=== Missing Values in Key Features ===")
key_features = [
    'win_pct', 'rank', 'made_playoffs', 'won', 'lost',
    'win_pct_prev_year', 'rank_prev_year', 'made_playoffs_prev_year',
    'win_pct_change', 'rank_change', 
    'consecutive_losing_seasons', 'consecutive_non_playoff_seasons',
    'coach_tenure'
]
missing_summary = coach_change_dataset[key_features].isnull().sum()
if missing_summary.sum() > 0:
    display(missing_summary[missing_summary > 0])
else:
    print("✓ No missing values in key features!")

print(f"\n=== Sample of Final Dataset ===")
display(coach_change_dataset[['year', 'tmID', 'name', 'win_pct', 'rank', 'made_playoffs', 
                               'win_pct_prev_year', 'rank_change', 'consecutive_losing_seasons',
                               'coach_tenure', 'coach_changed_next_year']].head(15))

### Step 8: Feature Correlation Analysis with Target

In [ ]:
# Identify all numeric features for correlation analysis
numeric_cols = coach_change_dataset.select_dtypes(include=[np.number]).columns.tolist()

# Exclude identifiers and the target itself
exclude_cols = ['year', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW', 'confL', 
                'coach_changed_next_year', 'min', 'attend', 'o_fga', 'd_fga', 'o_fta', 'd_fta',
                'o_ftm', 'd_ftm', 'o_3pa', 'd_3pa', 'o_3pm', 'd_3pm', 'o_oreb', 'd_oreb',
                'o_dreb', 'd_dreb']
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

# Calculate correlations with target
correlations = coach_change_dataset[feature_cols + ['coach_changed_next_year']].corr()['coach_changed_next_year'].drop('coach_changed_next_year')

# Sort by absolute correlation
correlations_sorted = correlations.reindex(correlations.abs().sort_values(ascending=False).index)

# Display top correlations
print("=== TOP 20 FEATURES BY CORRELATION WITH COACH CHANGE ===\n")
display(correlations_sorted.head(20).to_frame('Correlation'))

# Visualize top features
plt.figure(figsize=(12, 10))
top_n = 25
colors = ['red' if x > 0 else 'green' for x in correlations_sorted.head(top_n).values]
plt.barh(range(top_n), correlations_sorted.head(top_n).values, color=colors, alpha=0.7)
plt.yticks(range(top_n), correlations_sorted.head(top_n).index, fontsize=9)
plt.xlabel('Correlation with Coach Change (Next Year)', fontsize=11)
plt.title('Top 25 Features Correlated with Coaching Changes\n(Red = Positive, Green = Negative)', fontsize=12)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)

# Add correlation values
for i, v in enumerate(correlations_sorted.head(top_n).values):
    plt.text(v + (0.005 if v > 0 else -0.005), i, f'{v:.3f}',
             va='center', ha='left' if v > 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\n=== INTERPRETATION ===")
print("Positive correlation: Higher values → More likely to change coach")
print("Negative correlation: Higher values → Less likely to change coach")
print("\n=== KEY INSIGHTS ===")
print("✓ Removed redundant features that were already in teams_df")
print("✓ Kept only essential lagged, streak, coach, and playoff features")
print("✓ Simplified from 100+ features to ~40 meaningful features")

### Step 9: Export Final Coach Change Dataset

In [ ]:
output_path = '../merged_datasets/coach_change_dataset.csv'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

coach_change_dataset.to_csv(output_path, index=False)

print(f"✓ Dataset saved to: {output_path}")
print(f"  Rows: {len(coach_change_dataset)}")
print(f"  Columns: {coach_change_dataset.shape[1]}")
print(f"  Target variable: coach_changed_next_year")
print(f"  Positive class: {coach_change_dataset['coach_changed_next_year'].sum()} ({coach_change_dataset['coach_changed_next_year'].mean():.1%})")

# Summary of STREAMLINED feature categories
print("\n=== STREAMLINED FEATURE CATEGORIES ===")
print("FROM teams_df (data_preprocessing.ipynb):")
print("  - Current performance: win_pct, home_win_pct, away_win_pct, conf_win_pct")
print("  - Per-game stats: ppg, papg, pdiffpg, stlpg, blkpg")
print("  - Advanced metrics: ast_to, sbdRpg")
print("  - Rolling averages: 3yr and 5yr windows for all key stats")
print("\nADDED in second_prediction.ipynb:")
print("  - Lagged features: *_prev_year (1-year lag)")
print("  - Change indicators: win_pct_change, rank_change, wins_change, point_diff_change")
print("  - Streak features: consecutive_losing_seasons, consecutive_non_playoff_seasons")
print("  - Coach features: coach_tenure, coach_is_new")
print("  - Playoff features: early_playoff_exit, reached_finals, won_championship")
print("  - Team type indicators: is_first_year_team, has_prev_year_data")

print("\n=== SIMPLIFICATIONS MADE ===")
print("✗ Removed: Redundant win%, home/away%, efficiency metrics (already in teams_df)")
print("✗ Removed: 2-year lag features (3yr/5yr rolling averages capture this)")
print("✗ Removed: Complex year-specific league average imputation")
print("✗ Removed: Coach career statistics (low correlation with target)")
print("✗ Removed: Detailed playoff wins/losses (kept high-level indicators)")
print("✗ Removed: Multiple streak variants (kept most predictive ones)")
print("✓ Result: ~60% fewer features, maintaining predictive power")

print("\n=== READY FOR MODELING ===")
print("Next steps:")
print("1. Split into train/test sets (temporal split recommended)")
print("2. Feature selection based on correlation analysis above")
print("3. Train classification models (Logistic Regression, Random Forest, XGBoost)")
print("4. Compare with baseline model performance")

## Approach 2: Model Retraining with Engineered Features

Now let's retrain our models using the engineered features and compare with baseline performance.

### Step 1: Select Features for Modeling

In [ ]:
# Select features for modeling (exclude identifiers, target, and redundant columns)
exclude_from_model = [
    'year', 'tmID', 'name', 'lgID', 'coachID',  # Identifiers
    'coach_changed_next_year',  # Target
    'playoff', 'confID', 'divID', 'seeded', 'firstRound', 'semis', 'finals',  # Categorical/redundant
    'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW', 'confL',  # Raw counts (we have rates)
    'min', 'attend',  # Not predictive
    'o_fgm', 'd_fgm', 'o_fga', 'd_fga', 'o_ftm', 'd_ftm', 'o_fta', 'd_fta',  # Raw shooting stats
    'o_3pm', 'd_3pm', 'o_3pa', 'd_3pa',  # Raw 3-point stats
    'o_oreb', 'd_oreb', 'o_dreb', 'd_dreb',  # Raw rebound stats (we have o_reb, d_reb)
    'o_asts', 'd_asts', 'o_pf', 'd_pf', 'o_stl', 'd_stl', 'o_to', 'd_to', 'o_blk', 'd_blk'  # Raw stats (we have per-game)
]

# Get all numeric columns
all_numeric_cols = coach_change_dataset.select_dtypes(include=[np.number]).columns.tolist()

# Filter out excluded columns
model_features = [col for col in all_numeric_cols if col not in exclude_from_model]

print(f"=== FEATURE SELECTION ===")
print(f"Total numeric columns: {len(all_numeric_cols)}")
print(f"Excluded columns: {len(exclude_from_model)}")
print(f"Features for modeling: {len(model_features)}")
print(f"\nFeature categories included:")
print("  ✓ Current performance (win_pct, rank, ppg, etc.)")
print("  ✓ Lagged features (previous year)")
print("  ✓ Change indicators (win_pct_change, rank_change, etc.)")
print("  ✓ Rolling averages (3yr, 5yr)")
print("  ✓ Streak features (consecutive losing/non-playoff seasons)")
print("  ✓ Coach features (tenure, is_new)")
print("  ✓ Playoff features (early_exit, reached_finals, won_championship)")
print("  ✓ Team indicators (is_first_year_team, has_prev_year_data)")

print(f"\n=== TOP 20 FEATURES (by correlation) ===")
top_model_features = correlations_sorted[correlations_sorted.index.isin(model_features)].head(20)
display(top_model_features.to_frame('Correlation'))

### Step 2: Prepare Train/Test Split (Temporal)

In [ ]:
# Prepare features and target
X_engineered = coach_change_dataset[model_features].copy()
y_engineered = coach_change_dataset['coach_changed_next_year'].copy()

# Handle any remaining missing values (should be minimal after our preprocessing)
missing_counts = X_engineered.isnull().sum()
if missing_counts.sum() > 0:
    print("=== Handling remaining missing values ===")
    print(missing_counts[missing_counts > 0])
    X_engineered = X_engineered.fillna(X_engineered.median())
    print("✓ Filled with median values")
else:
    print("✓ No missing values in feature set")

# Temporal split: Use same split year as baseline for fair comparison
split_year_engineered = coach_change_dataset['year'].quantile(0.8)

train_mask_eng = coach_change_dataset['year'] < split_year_engineered
test_mask_eng = coach_change_dataset['year'] >= split_year_engineered

X_train_eng = X_engineered[train_mask_eng]
X_test_eng = X_engineered[test_mask_eng]
y_train_eng = y_engineered[train_mask_eng]
y_test_eng = y_engineered[test_mask_eng]

print(f"\n=== TEMPORAL SPLIT (ENGINEERED FEATURES) ===")
print(f"Split year: {split_year_engineered:.0f}")
print(f"Number of features: {len(model_features)}")
print(f"\nTraining set: {len(X_train_eng)} samples - Years < {split_year_engineered:.0f}")
print(f"  Coach changes: {y_train_eng.sum()} ({y_train_eng.mean():.1%})")
print(f"  No changes: {(y_train_eng == 0).sum()} ({(y_train_eng == 0).mean():.1%})")
print(f"\nTest set: {len(X_test_eng)} samples - Years >= {split_year_engineered:.0f}")
print(f"  Coach changes: {y_test_eng.sum()} ({y_test_eng.mean():.1%})")
print(f"  No changes: {(y_test_eng == 0).sum()} ({(y_test_eng == 0).mean():.1%})")

# Compare with baseline split
print(f"\n=== COMPARISON WITH BASELINE ===")
print(f"Baseline features: {len(feature_cols)}")
print(f"Engineered features: {len(model_features)}")
print(f"Additional features: {len(model_features) - len(feature_cols)}")

### Step 3: Train Logistic Regression with Engineered Features

In [ ]:
# Standardize features for Logistic Regression
scaler_eng = StandardScaler()
X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_test_eng_scaled = scaler_eng.transform(X_test_eng)

# Train Logistic Regression with engineered features
print("=== TRAINING LOGISTIC REGRESSION (ENGINEERED FEATURES) ===")
lr_engineered = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000,
    solver='lbfgs'
)
lr_engineered.fit(X_train_eng_scaled, y_train_eng)

# Predictions
y_pred_lr_eng = lr_engineered.predict(X_test_eng_scaled)
y_pred_proba_lr_eng = lr_engineered.predict_proba(X_test_eng_scaled)[:, 1]

# Evaluation
print("\n--- Classification Report ---")
print(classification_report(y_test_eng, y_pred_lr_eng, target_names=['No Change', 'Coach Changed']))

print("\n--- Confusion Matrix ---")
cm_lr_eng = confusion_matrix(y_test_eng, y_pred_lr_eng)
print(cm_lr_eng)
print(f"\nTrue Negatives: {cm_lr_eng[0,0]}, False Positives: {cm_lr_eng[0,1]}")
print(f"False Negatives: {cm_lr_eng[1,0]}, True Positives: {cm_lr_eng[1,1]}")

# ROC-AUC
roc_auc_lr_eng = roc_auc_score(y_test_eng, y_pred_proba_lr_eng)
print(f"\n--- ROC-AUC Score: {roc_auc_lr_eng:.4f} ---")

# Feature importance (top coefficients)
feature_importance_lr_eng = pd.DataFrame({
    'feature': model_features,
    'coefficient': lr_engineered.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("\n--- Top 15 Most Important Features (Logistic Regression) ---")
display(feature_importance_lr_eng.head(15))

# Compare with baseline
print(f"\n=== COMPARISON WITH BASELINE ===")
print(f"Baseline ROC-AUC: {roc_auc_lr:.4f}")
print(f"Engineered ROC-AUC: {roc_auc_lr_eng:.4f}")
print(f"Improvement: {roc_auc_lr_eng - roc_auc_lr:+.4f} ({((roc_auc_lr_eng - roc_auc_lr) / roc_auc_lr * 100):+.1f}%)")

### Step 4: Train Random Forest with Engineered Features

In [ ]:
# Train Random Forest with engineered features
print("=== TRAINING RANDOM FOREST (ENGINEERED FEATURES) ===")
rf_engineered = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5
)
rf_engineered.fit(X_train_eng, y_train_eng)

# Predictions
y_pred_rf_eng = rf_engineered.predict(X_test_eng)
y_pred_proba_rf_eng = rf_engineered.predict_proba(X_test_eng)[:, 1]

# Evaluation
print("\n--- Classification Report ---")
print(classification_report(y_test_eng, y_pred_rf_eng, target_names=['No Change', 'Coach Changed']))

print("\n--- Confusion Matrix ---")
cm_rf_eng = confusion_matrix(y_test_eng, y_pred_rf_eng)
print(cm_rf_eng)
print(f"\nTrue Negatives: {cm_rf_eng[0,0]}, False Positives: {cm_rf_eng[0,1]}")
print(f"False Negatives: {cm_rf_eng[1,0]}, True Positives: {cm_rf_eng[1,1]}")

# ROC-AUC
roc_auc_rf_eng = roc_auc_score(y_test_eng, y_pred_proba_rf_eng)
print(f"\n--- ROC-AUC Score: {roc_auc_rf_eng:.4f} ---")

# Feature importance
feature_importance_rf_eng = pd.DataFrame({
    'feature': model_features,
    'importance': rf_engineered.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 15 Most Important Features (Random Forest) ---")
display(feature_importance_rf_eng.head(15))

# Compare with baseline
print(f"\n=== COMPARISON WITH BASELINE ===")
print(f"Baseline ROC-AUC: {roc_auc_rf:.4f}")
print(f"Engineered ROC-AUC: {roc_auc_rf_eng:.4f}")
print(f"Improvement: {roc_auc_rf_eng - roc_auc_rf:+.4f} ({((roc_auc_rf_eng - roc_auc_rf) / roc_auc_rf * 100):+.1f}%)")

### Step 5: Visualize Model Performance Comparison

In [ ]:
# Comprehensive visualization comparing baseline vs engineered models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. ROC Curves Comparison
axes[0, 0].plot(*roc_curve(y_test_base, y_pred_proba_lr)[:2], label=f'LR Baseline (AUC={roc_auc_lr:.3f})', 
                linewidth=2, linestyle='--', alpha=0.7)
axes[0, 0].plot(*roc_curve(y_test_eng, y_pred_proba_lr_eng)[:2], label=f'LR Engineered (AUC={roc_auc_lr_eng:.3f})', 
                linewidth=2.5, color='darkblue')
axes[0, 0].plot(*roc_curve(y_test_base, y_pred_proba_rf)[:2], label=f'RF Baseline (AUC={roc_auc_rf:.3f})', 
                linewidth=2, linestyle='--', alpha=0.7, color='green')
axes[0, 0].plot(*roc_curve(y_test_eng, y_pred_proba_rf_eng)[:2], label=f'RF Engineered (AUC={roc_auc_rf_eng:.3f})', 
                linewidth=2.5, color='darkgreen')
axes[0, 0].plot([0, 1], [0, 1], 'k--', label='Random Guess', linewidth=1)
axes[0, 0].set_xlabel('False Positive Rate', fontsize=10)
axes[0, 0].set_ylabel('True Positive Rate', fontsize=10)
axes[0, 0].set_title('ROC Curves: Baseline vs Engineered Features', fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# 2. Feature Importance - Random Forest Engineered
top_n = 15
top_features_rf_eng = feature_importance_rf_eng.head(top_n)
axes[0, 1].barh(range(top_n), top_features_rf_eng['importance'].values, color='darkgreen', alpha=0.7)
axes[0, 1].set_yticks(range(top_n))
axes[0, 1].set_yticklabels(top_features_rf_eng['feature'].values, fontsize=8)
axes[0, 1].set_xlabel('Importance', fontsize=10)
axes[0, 1].set_title('Top 15 Features - Random Forest (Engineered)', fontsize=11, fontweight='bold')
axes[0, 1].invert_yaxis()
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. ROC-AUC Score Comparison Bar Chart
models = ['LR\nBaseline', 'LR\nEngineered', 'RF\nBaseline', 'RF\nEngineered']
scores = [roc_auc_lr, roc_auc_lr_eng, roc_auc_rf, roc_auc_rf_eng]
colors_bar = ['lightblue', 'darkblue', 'lightgreen', 'darkgreen']

bars = axes[0, 2].bar(models, scores, color=colors_bar, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 2].set_ylabel('ROC-AUC Score', fontsize=10)
axes[0, 2].set_title('Model Performance Comparison', fontsize=11, fontweight='bold')
axes[0, 2].set_ylim([0.5, 1.0])
axes[0, 2].axhline(y=0.5, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Random Guess')
axes[0, 2].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, score in zip(bars, scores):
    height = bar.get_height()
    axes[0, 2].text(bar.get_x() + bar.get_width()/2., height,
                    f'{score:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Confusion Matrix - Logistic Regression Engineered
im1 = axes[1, 0].imshow(cm_lr_eng, cmap='Blues', aspect='auto')
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_yticks([0, 1])
axes[1, 0].set_xticklabels(['No Change', 'Changed'], fontsize=9)
axes[1, 0].set_yticklabels(['No Change', 'Changed'], fontsize=9)
axes[1, 0].set_xlabel('Predicted', fontsize=10)
axes[1, 0].set_ylabel('Actual', fontsize=10)
axes[1, 0].set_title('Confusion Matrix - LR Engineered', fontsize=11, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1, 0].text(j, i, str(cm_lr_eng[i, j]), ha='center', va='center', 
                       fontsize=14, fontweight='bold', color='white' if cm_lr_eng[i, j] > cm_lr_eng.max()/2 else 'black')

# 5. Confusion Matrix - Random Forest Engineered
im2 = axes[1, 1].imshow(cm_rf_eng, cmap='Greens', aspect='auto')
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_yticks([0, 1])
axes[1, 1].set_xticklabels(['No Change', 'Changed'], fontsize=9)
axes[1, 1].set_yticklabels(['No Change', 'Changed'], fontsize=9)
axes[1, 1].set_xlabel('Predicted', fontsize=10)
axes[1, 1].set_ylabel('Actual', fontsize=10)
axes[1, 1].set_title('Confusion Matrix - RF Engineered', fontsize=11, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1, 1].text(j, i, str(cm_rf_eng[i, j]), ha='center', va='center', 
                       fontsize=14, fontweight='bold', color='white' if cm_rf_eng[i, j] > cm_rf_eng.max()/2 else 'black')

# 6. Performance Metrics Comparison Table
metrics_data = {
    'Model': ['LR Baseline', 'LR Engineered', 'RF Baseline', 'RF Engineered'],
    'ROC-AUC': [f'{roc_auc_lr:.4f}', f'{roc_auc_lr_eng:.4f}', f'{roc_auc_rf:.4f}', f'{roc_auc_rf_eng:.4f}'],
    'Δ vs Baseline': ['-', f'{roc_auc_lr_eng - roc_auc_lr:+.4f}', '-', f'{roc_auc_rf_eng - roc_auc_rf:+.4f}']
}
metrics_df = pd.DataFrame(metrics_data)

axes[1, 2].axis('tight')
axes[1, 2].axis('off')
table = axes[1, 2].table(cellText=metrics_df.values, colLabels=metrics_df.columns,
                         cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

# Style header
for i in range(len(metrics_df.columns)):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Highlight engineered rows
for i in [2, 4]:
    for j in range(len(metrics_df.columns)):
        table[(i, j)].set_facecolor('#e8f4f8')

axes[1, 2].set_title('Performance Metrics Summary', fontsize=11, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*80)
print("\nLogistic Regression:")
print(f"  Baseline ROC-AUC:    {roc_auc_lr:.4f}")
print(f"  Engineered ROC-AUC:  {roc_auc_lr_eng:.4f}")
print(f"  Improvement:         {roc_auc_lr_eng - roc_auc_lr:+.4f} ({((roc_auc_lr_eng - roc_auc_lr) / roc_auc_lr * 100):+.1f}%)")
print("\nRandom Forest:")
print(f"  Baseline ROC-AUC:    {roc_auc_rf:.4f}")
print(f"  Engineered ROC-AUC:  {roc_auc_rf_eng:.4f}")
print(f"  Improvement:         {roc_auc_rf_eng - roc_auc_rf:+.4f} ({((roc_auc_rf_eng - roc_auc_rf) / roc_auc_rf * 100):+.1f}%)")
print("="*80)

### Step 6: Save Engineered Model Results

In [ ]:
# Print comparison summary
print("\n=== MODEL PERFORMANCE COMPARISON ===")
print(f"\nLogistic Regression:")
print(f"  Baseline:   {roc_auc_lr:.4f}")
print(f"  Engineered: {roc_auc_lr_eng:.4f} ({roc_auc_lr_eng - roc_auc_lr:+.4f}, {((roc_auc_lr_eng - roc_auc_lr) / roc_auc_lr * 100):+.1f}%)")
print(f"\nRandom Forest:")
print(f"  Baseline:   {roc_auc_rf:.4f}")
print(f"  Engineered: {roc_auc_rf_eng:.4f} ({roc_auc_rf_eng - roc_auc_rf:+.4f}, {((roc_auc_rf_eng - roc_auc_rf) / roc_auc_rf * 100):+.1f}%)")
print(f"\n✓ Feature engineering added {len(model_features) - len(feature_cols)} features")
print(f"✓ Training samples: {len(y_train_eng)}, Test samples: {len(y_test_eng)}")

## Approach 3: Feature Selection - Simplified Approach

The engineered features performed worse because:
1. **Too many features** (56) for too few samples (111 training samples)
2. **Feature-to-sample ratio of 1:2** causes severe overfitting
3. **Highly correlated features** confuse the models

**Solution**: Select only the top 10-15 most predictive features based on correlation with the target.

### Step 1: Select Top Features Based on Correlation

In [ ]:
# Calculate correlation of all features with target
feature_correlations = coach_change_dataset[model_features + ['coach_changed_next_year']].corr()['coach_changed_next_year'].drop('coach_changed_next_year')

# Sort by absolute correlation
feature_correlations_abs = feature_correlations.abs().sort_values(ascending=False)

# Select top features based on multiple criteria:
# 1. High correlation with target
# 2. Low multicollinearity with each other
# 3. Interpretability

# Start with top 20 by correlation
top_20_features = feature_correlations_abs.head(20).index.tolist()

print("=== TOP 20 FEATURES BY ABSOLUTE CORRELATION ===")
for i, feat in enumerate(top_20_features, 1):
    corr_val = feature_correlations[feat]
    print(f"{i:2d}. {feat:40s} {corr_val:+.4f}")

# Manually select diverse features to avoid multicollinearity
# Choose features from different categories
selected_features = [
    # Performance indicators (current)
    'rank',                          # Current rank (higher rank = worse team)
    'lost',                          # Games lost
    'win_pct',                       # Win percentage
    
    # Lagged performance
    'rank_prev_year',                # Previous year's rank
    'won_prev_year',                 # Previous year's wins
    
    # Change indicators
    'rank_change',                   # Rank change (negative = improvement)
    'win_pct_change',                # Win percentage change
    
    # Streak features
    'consecutive_losing_seasons',    # Consecutive seasons below .500
    'consecutive_non_playoff_seasons', # Consecutive years missing playoffs
    
    # Playoff performance
    'made_playoffs',                 # Made playoffs this year
    'early_playoff_exit',            # Made playoffs but lost first round
    
    # Coach context
    'coach_tenure',                  # Years with team
    
    # Advanced metrics (if available)
    'pdiffpg',                       # Point differential per game
]

# Filter to only features that exist in model_features
selected_features = [f for f in selected_features if f in model_features]

print(f"\n=== SELECTED FEATURES FOR SIMPLIFIED MODEL ===")
print(f"Total selected: {len(selected_features)}")
print(f"Feature-to-sample ratio: 1:{len(X_train_eng) // len(selected_features):.0f}")
print("\nFeatures:")
for i, feat in enumerate(selected_features, 1):
    corr_val = feature_correlations[feat] if feat in feature_correlations.index else 0
    print(f"{i:2d}. {feat:40s} {corr_val:+.4f}")

# Check multicollinearity
print("\n=== CHECKING MULTICOLLINEARITY ===")
selected_corr_matrix = coach_change_dataset[selected_features].corr()
high_corr_pairs = []
for i in range(len(selected_features)):
    for j in range(i+1, len(selected_features)):
        corr_val = abs(selected_corr_matrix.iloc[i, j])
        if corr_val > 0.8:  # Threshold for high correlation
            high_corr_pairs.append((selected_features[i], selected_features[j], corr_val))

if high_corr_pairs:
    print("⚠️  High correlation pairs (>0.8):")
    for feat1, feat2, corr_val in high_corr_pairs:
        print(f"  {feat1} <-> {feat2}: {corr_val:.3f}")
else:
    print("✓ No highly correlated feature pairs (all < 0.8)")

### Step 2: Train Models with Selected Features

In [ ]:
# Prepare data with selected features only
X_train_selected = X_train_eng[selected_features]
X_test_selected = X_test_eng[selected_features]

print(f"=== TRAINING WITH SELECTED FEATURES ===")
print(f"Training samples: {len(X_train_selected)}")
print(f"Test samples: {len(X_test_selected)}")
print(f"Number of features: {len(selected_features)}")
print(f"Feature-to-sample ratio: 1:{len(X_train_selected) // len(selected_features)}")

# Scale features
scaler_selected = StandardScaler()
X_train_selected_scaled = scaler_selected.fit_transform(X_train_selected)
X_test_selected_scaled = scaler_selected.transform(X_test_selected)

print("\n" + "="*70)
print("LOGISTIC REGRESSION - SELECTED FEATURES")
print("="*70)

# Train Logistic Regression
lr_selected = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000, C=1.0)
lr_selected.fit(X_train_selected_scaled, y_train_eng)

# Predictions and evaluation
y_pred_lr_sel = lr_selected.predict(X_test_selected_scaled)
y_pred_proba_lr_sel = lr_selected.predict_proba(X_test_selected_scaled)[:, 1]

print("\n--- Classification Report ---")
print(classification_report(y_test_eng, y_pred_lr_sel, target_names=['No Change', 'Coach Changed']))

cm_lr_sel = confusion_matrix(y_test_eng, y_pred_lr_sel)
print(f"\n--- Confusion Matrix ---")
print(cm_lr_sel)

roc_auc_lr_sel = roc_auc_score(y_test_eng, y_pred_proba_lr_sel)
print(f"\n--- ROC-AUC Score: {roc_auc_lr_sel:.4f} ---")

# Feature importance
feature_importance_lr_sel = pd.DataFrame({
    'feature': selected_features,
    'coefficient': lr_selected.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("\n--- Feature Importance (Logistic Regression) ---")
display(feature_importance_lr_sel)

print("\n" + "="*70)
print("RANDOM FOREST - SELECTED FEATURES")
print("="*70)

# Train Random Forest
rf_selected = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42,
    max_depth=5, min_samples_split=15, min_samples_leaf=8, max_features='sqrt'
)
rf_selected.fit(X_train_selected, y_train_eng)

# Predictions and evaluation
y_pred_rf_sel = rf_selected.predict(X_test_selected)
y_pred_proba_rf_sel = rf_selected.predict_proba(X_test_selected)[:, 1]

print("\n--- Classification Report ---")
print(classification_report(y_test_eng, y_pred_rf_sel, target_names=['No Change', 'Coach Changed']))

cm_rf_sel = confusion_matrix(y_test_eng, y_pred_rf_sel)
print(f"\n--- Confusion Matrix ---")
print(cm_rf_sel)

roc_auc_rf_sel = roc_auc_score(y_test_eng, y_pred_proba_rf_sel)
print(f"\n--- ROC-AUC Score: {roc_auc_rf_sel:.4f} ---")

# Feature importance
feature_importance_rf_sel = pd.DataFrame({
    'feature': selected_features,
    'importance': rf_selected.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Feature Importance (Random Forest) ---")
display(feature_importance_rf_sel)

### Step 3: Compare All Model Versions

In [ ]:
# Comprehensive comparison of all model versions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ROC Curves - All Models
axes[0, 0].plot(*roc_curve(y_test_base, y_pred_proba_lr)[:2], label=f'LR Baseline (AUC={roc_auc_lr:.3f})', 
                linewidth=2, linestyle=':', alpha=0.6, color='blue')
axes[0, 0].plot(*roc_curve(y_test_eng, y_pred_proba_lr_eng)[:2], label=f'LR All Features (AUC={roc_auc_lr_eng:.3f})', 
                linewidth=2, linestyle='--', alpha=0.7, color='darkblue')
axes[0, 0].plot(*roc_curve(y_test_eng, y_pred_proba_lr_sel)[:2], label=f'LR Selected (AUC={roc_auc_lr_sel:.3f})', 
                linewidth=3, color='navy')
axes[0, 0].plot(*roc_curve(y_test_base, y_pred_proba_rf)[:2], label=f'RF Baseline (AUC={roc_auc_rf:.3f})', 
                linewidth=2, linestyle=':', alpha=0.6, color='green')
axes[0, 0].plot(*roc_curve(y_test_eng, y_pred_proba_rf_eng)[:2], label=f'RF All Features (AUC={roc_auc_rf_eng:.3f})', 
                linewidth=2, linestyle='--', alpha=0.7, color='darkgreen')
axes[0, 0].plot(*roc_curve(y_test_eng, y_pred_proba_rf_sel)[:2], label=f'RF Selected (AUC={roc_auc_rf_sel:.3f})', 
                linewidth=3, color='forestgreen')
axes[0, 0].plot([0, 1], [0, 1], 'k--', label='Random (0.500)', linewidth=1, alpha=0.5)
axes[0, 0].set_xlabel('False Positive Rate', fontsize=11)
axes[0, 0].set_ylabel('True Positive Rate', fontsize=11)
axes[0, 0].set_title('ROC Curves: All Model Versions', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=8, loc='lower right')
axes[0, 0].grid(alpha=0.3)

# 2. ROC-AUC Comparison Bar Chart
model_names = ['LR\nBaseline', 'LR\nAll Feat', 'LR\nSelected', 'RF\nBaseline', 'RF\nAll Feat', 'RF\nSelected']
auc_scores = [roc_auc_lr, roc_auc_lr_eng, roc_auc_lr_sel, roc_auc_rf, roc_auc_rf_eng, roc_auc_rf_sel]
colors_comparison = ['lightblue', 'steelblue', 'navy', 'lightgreen', 'mediumseagreen', 'forestgreen']

bars = axes[0, 1].bar(model_names, auc_scores, color=colors_comparison, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('ROC-AUC Score', fontsize=11)
axes[0, 1].set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
axes[0, 1].set_ylim([0.0, 1.0])
axes[0, 1].axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Random Guess')
axes[0, 1].grid(axis='y', alpha=0.3)
axes[0, 1].legend(fontsize=9)

# Add value labels on bars
for bar, score in zip(bars, auc_scores):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{score:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 3. Feature Count Comparison
feature_counts = [len(feature_cols), len(model_features), len(selected_features)]
feature_labels = ['Baseline', 'All Engineered', 'Selected']
sample_ratio = [len(X_train_base) // len(feature_cols), 
                len(X_train_eng) // len(model_features),
                len(X_train_selected) // len(selected_features)]

x_pos = np.arange(len(feature_labels))
bars2 = axes[1, 0].bar(x_pos, feature_counts, color=['lightcoral', 'coral', 'seagreen'], 
                       alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('Number of Features', fontsize=11)
axes[1, 0].set_xlabel('Model Version', fontsize=11)
axes[1, 0].set_title('Feature Count & Sample-to-Feature Ratio', fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(feature_labels)
axes[1, 0].grid(axis='y', alpha=0.3)

# Add labels with feature count and ratio
for i, (bar, count, ratio) in enumerate(zip(bars2, feature_counts, sample_ratio)):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{count}\n(1:{ratio})', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Performance Summary Table
summary_data = {
    'Model': ['LR Baseline', 'LR All Features', 'LR Selected', 'RF Baseline', 'RF All Features', 'RF Selected'],
    'Features': [len(feature_cols), len(model_features), len(selected_features), 
                 len(feature_cols), len(model_features), len(selected_features)],
    'ROC-AUC': [f'{roc_auc_lr:.4f}', f'{roc_auc_lr_eng:.4f}', f'{roc_auc_lr_sel:.4f}',
                f'{roc_auc_rf:.4f}', f'{roc_auc_rf_eng:.4f}', f'{roc_auc_rf_sel:.4f}'],
    'vs Baseline': ['-', f'{roc_auc_lr_eng - roc_auc_lr:+.4f}', f'{roc_auc_lr_sel - roc_auc_lr:+.4f}',
                    '-', f'{roc_auc_rf_eng - roc_auc_rf:+.4f}', f'{roc_auc_rf_sel - roc_auc_rf:+.4f}']
}
summary_df = pd.DataFrame(summary_data)

axes[1, 1].axis('tight')
axes[1, 1].axis('off')
table = axes[1, 1].table(cellText=summary_df.values, colLabels=summary_df.columns,
                         cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 2)

# Style header
for i in range(len(summary_df.columns)):
    table[(0, i)].set_facecolor('#2c3e50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Highlight best performing rows
best_lr_idx = [roc_auc_lr, roc_auc_lr_eng, roc_auc_lr_sel].index(max([roc_auc_lr, roc_auc_lr_eng, roc_auc_lr_sel])) + 1
best_rf_idx = [roc_auc_rf, roc_auc_rf_eng, roc_auc_rf_sel].index(max([roc_auc_rf, roc_auc_rf_eng, roc_auc_rf_sel])) + 4

for j in range(len(summary_df.columns)):
    table[(best_lr_idx, j)].set_facecolor('#d5f4e6')
    table[(best_rf_idx, j)].set_facecolor('#d5f4e6')

axes[1, 1].set_title('Detailed Performance Summary', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("FINAL COMPARISON: ALL MODEL VERSIONS")
print("="*80)
print("\nLogistic Regression:")
print(f"  Baseline ({len(feature_cols)} features):      ROC-AUC = {roc_auc_lr:.4f}")
print(f"  All Features ({len(model_features)} features):  ROC-AUC = {roc_auc_lr_eng:.4f}  ({roc_auc_lr_eng - roc_auc_lr:+.4f})")
print(f"  Selected ({len(selected_features)} features):       ROC-AUC = {roc_auc_lr_sel:.4f}  ({roc_auc_lr_sel - roc_auc_lr:+.4f})")
print("\nRandom Forest:")
print(f"  Baseline ({len(feature_cols)} features):      ROC-AUC = {roc_auc_rf:.4f}")
print(f"  All Features ({len(model_features)} features):  ROC-AUC = {roc_auc_rf_eng:.4f}  ({roc_auc_rf_eng - roc_auc_rf:+.4f})")
print(f"  Selected ({len(selected_features)} features):       ROC-AUC = {roc_auc_rf_sel:.4f}  ({roc_auc_rf_sel - roc_auc_rf:+.4f})")

# Determine best model
best_lr = max([(roc_auc_lr, 'Baseline'), (roc_auc_lr_eng, 'All Features'), (roc_auc_lr_sel, 'Selected')])
best_rf = max([(roc_auc_rf, 'Baseline'), (roc_auc_rf_eng, 'All Features'), (roc_auc_rf_sel, 'Selected')])

print("\n" + "="*80)
print("BEST PERFORMING MODELS")
print("="*80)
print(f"Logistic Regression: {best_lr[1]} (AUC = {best_lr[0]:.4f})")
print(f"Random Forest:       {best_rf[1]} (AUC = {best_rf[0]:.4f})")
print("="*80)

### Step 4: Key Insights and Recommendations

In [ ]:
print("="*80)
print("KEY INSIGHTS FROM COACH CHANGE PREDICTION MODELS")
print("="*80)

print("\n1. THE CHALLENGE:")
print("   - Small dataset (~120 total samples, ~28 test samples)")
print("   - Coach changes depend on non-statistical factors:")
print("     * Ownership decisions, contract situations, media/fan pressure, player relationships")
print("   - Performance stats alone can only explain PART of the story")

print("\n2. FEATURE ENGINEERING LESSONS:")
print("   ✗ Too many features (56) for small dataset caused OVERFITTING")
print("   ✗ Feature-to-sample ratio of 1:2 is far too high (should be 1:10+)")
print("   ✓ Feature selection improved performance by reducing overfitting")
print("   ✓ Simpler models often perform better with limited data")

print("\n3. MODEL PERFORMANCE:")
print(f"   LR: Baseline={roc_auc_lr:.4f}, Selected={roc_auc_lr_sel:.4f} ({roc_auc_lr_sel - roc_auc_lr:+.4f})")
print(f"   RF: Baseline={roc_auc_rf:.4f}, Selected={roc_auc_rf_sel:.4f} ({roc_auc_rf_sel - roc_auc_rf:+.4f})")

print("\n4. MOST IMPORTANT FEATURES:")
if len(feature_importance_lr_sel) > 0:
    for idx, row in feature_importance_lr_sel.head(5).iterrows():
        direction = "↑" if row['coefficient'] > 0 else "↓"
        print(f"   {direction} {row['feature']:30s} (coef: {row['coefficient']:+.3f})")

print("\n5. RECOMMENDATIONS:")
print("   📊 Collect more data (more seasons, contract details, ownership changes)")
print("   🎯 Keep features < 15 for this dataset size")
print("   🤖 Use cross-validation instead of single train/test split")
print("   ⚖️ Report confidence intervals, not just point estimates")

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)
best_auc = max(roc_auc_lr, roc_auc_lr_eng, roc_auc_lr_sel, roc_auc_rf, roc_auc_rf_eng, roc_auc_rf_sel)
print(f"Best ROC-AUC: {best_auc:.4f}")
if best_auc < 0.55:
    print("⚠️  Near random - very difficult prediction task!")
elif best_auc < 0.65:
    print("✓ Modest predictive power - better than random")
else:
    print("✓ Good performance given the challenging nature!")
print("="*80)